In [1]:
import pandas as pd
import numpy as np
from tkinter import filedialog
import tkinter as tk
import os

In [20]:


ROOT = "../Resultados/"
# Crear ventana temporal para seleccionar archivo
root = tk.Tk()
root.withdraw()  # Ocultar ventana principal

# Abrir diálogo para seleccionar archivo CSV
file_path = filedialog.askopenfilename(
    initialdir=ROOT,
    title="Seleccionar archivo CSV",
    filetypes=(("CSV files", "*.csv"), ("All files", "*.*"))
)

# Extraer el directorio del archivo seleccionado
# dir_path = os.path.dirname(file_path) if file_path else ROOT + "classification_exclude_prod_nested"
df_results = pd.read_csv(file_path)


In [21]:
# Encontrar el mejor modelo por cada N_Classes según Accuracy_Test
best_results = df_results.loc[df_results.groupby('N_Classes')['Accuracy_Test'].idxmax()]
best_results = best_results[['N_Classes', 'Model', 'Accuracy_Test', 'F1_Test', 'Best_Params']]
best_results = best_results.sort_values('Accuracy_Test', ascending=False)
best_results

,N_Classes,Model,Accuracy_Test,F1_Test,Best_Params
12,3_Nitrogen,XGB,0.796512,0.796512,"{'clf__colsample_bytree': 1.0, 'clf__learning_..."
13,3_Phosphorus,XGB,0.792151,0.792151,"{'clf__colsample_bytree': 1.0, 'clf__learning_..."
14,3_Potassium,XGB,0.789244,0.789244,"{'clf__colsample_bytree': 1.0, 'clf__learning_..."


## Prueba para 1 Modelo

In [2]:
from utils import *

In [11]:
# Crear ventana temporal para seleccionar archivo
root = tk.Tk()
root.withdraw()  # Ocultar ventana principal

PATH_MODEL = filedialog.askopenfilename(
    title="Seleccione el archivo del modelo",
    filetypes=[("Pickle files", "*.pkl"), ("All files", "*.*")]
)

if PATH_MODEL:
    with open(PATH_MODEL, 'rb') as pkl_file:
        all_results = pickle.load(pkl_file)
    print(f"Archivo cargado: {PATH_MODEL}")
else:
    print("No se seleccionó ningún archivo")

#obtener el path del directorio del modelo
model_dir = os.path.dirname(PATH_MODEL)
print(f"Directorio del modelo: {model_dir}")

# obtener nombre de ultima carpeta del path del modelo
model_name = os.path.basename(model_dir)
print(f"Nombre del modelo: {model_name}")


Archivo cargado: D:/Estudio/OneDrive - Universidad de Antioquia/Estudio/PAI/Codigo/Code_Quindio/Resultados/classification_exclude_prod_all_models_NPK/class_results_individual_elements.pkl
Directorio del modelo: D:/Estudio/OneDrive - Universidad de Antioquia/Estudio/PAI/Codigo/Code_Quindio/Resultados/classification_exclude_prod_all_models_NPK
Nombre del modelo: classification_exclude_prod_all_models_NPK


In [12]:
folder_model_name = os.path.basename(model_dir)
if "cuartiles" in folder_model_name:
    CFG.cuartiles_train = True
    CFG.individual_train = False
    n_modelos = 1
    labels_map = {0: "Q1", 1: "Q4"}
else:
    CFG.cuartiles_train = False
    CFG.individual_train = True
    n_modelos = 3
    labels_map = {0: "Not Adequated", 1: "Adequated", 2: "Excess"}

In [13]:
all_results['RF'][0]['classification_report']

{'0.0': {'precision': 0.7860892388451444,
  'recall': 0.7871222076215506,
  'f1-score': 0.7866053841103086,
  'support': 761.0},
 '1.0': {'precision': 0.7749029754204398,
  'recall': 0.7850589777195282,
  'f1-score': 0.7799479166666666,
  'support': 763.0},
 '2.0': {'precision': 0.7859973579920739,
  'recall': 0.7747395833333334,
  'f1-score': 0.780327868852459,
  'support': 768.0},
 'accuracy': 0.7822862129144852,
 'macro avg': {'precision': 0.7823298574192193,
  'recall': 0.7823069228914706,
  'f1-score': 0.7822937232098114,
  'support': 2292.0},
 'weighted avg': {'precision': 0.7823345776373749,
  'recall': 0.7822862129144852,
  'f1-score': 0.7822856723400087,
  'support': 2292.0}}

In [14]:

# Build classification report table: rows = (Class, Metric), columns = algorithms
algorithms = list(all_results.keys())
metrics_map = {'precision': 'Precision', 'recall': 'Recall', 'f1-score': 'F1-score'}

# Get class labels from the first algorithm's report
first_report = all_results[algorithms[0]][0]['classification_report']
class_labels = [k for k in first_report.keys() if k not in ['accuracy', 'macro avg', 'weighted avg']]

for model in range(n_modelos):
    rows = []
    index_tuples = []

    for class_label in class_labels:
        for metric_key, metric_name in metrics_map.items():
            row = {}
            for algo in algorithms:
                report = all_results[algo][model]['classification_report']
                row[algo] = round(report[str(class_label)][metric_key], 4)
            rows.append(row)
            aux_label = labels_map[(float(class_label))]
            index_tuples.append((f'Class {aux_label}', metric_name))

    # Overall Performance rows
    overall_metrics = [
        ('Accuracy',            lambda r: round(r['accuracy'], 4)),
        ('Macro Avg F1-score',  lambda r: round(r['macro avg']['f1-score'], 4)),
        ('Weighted Avg F1-score', lambda r: round(r['weighted avg']['f1-score'], 4)),
    ]
    for metric_name, extractor in overall_metrics:
        row = {}
        for algo in algorithms:
            report = all_results[algo][model]['classification_report']
            row[algo] = extractor(report)
        rows.append(row)
        index_tuples.append(('Overall Performance', metric_name))

    index = pd.MultiIndex.from_tuples(index_tuples, names=['', 'Metric'])
    df_report = pd.DataFrame(rows, index=index, columns=algorithms)
    df_report.index.names = ['', '']
    print(all_results[algo][model]['n_clases'])
    display(df_report)


3_Nitrogen


RF     SVM     KNN     MLP  \
                                                                            
Class Not Adequated Precision              0.7861  0.7073  0.6574  0.7340   
                    Recall                 0.7871  0.7240  0.7766  0.7254   
                    F1-score               0.7866  0.7156  0.7120  0.7297   
Class Adequated     Precision              0.7749  0.7483  0.7645  0.7374   
                    Recall                 0.7851  0.7326  0.6933  0.7287   
                    F1-score               0.7799  0.7404  0.7271  0.7330   
Class Excess        Precision              0.7860  0.7402  0.7618  0.7405   
                    Recall                 0.7747  0.7383  0.6953  0.7578   
                    F1-score               0.7803  0.7392  0.7270  0.7490   
Overall Performance Accuracy               0.7823  0.7317  0.7216  0.7373   
                    Macro Avg F1-score     0.7823  0.7317  0.7221  0.7372   
                    Weighted Avg F1-score  0.7823  0.7318  0.7221  0.7373   

                                              XGB  
                                                   
Class Not Adequated Precision              0.7859  
                    Recall                 0.7911  
                    F1-score               0.7885  
Class Adequated     Precision              0.8005  
                    Recall                 0.7890  
                    F1-score               0.7947  
Class Excess        Precision              0.7894  
                    Recall                 0.7956  
                    F1-score               0.7925  
Overall Performance Accuracy               0.7919  
                    Macro Avg F1-score     0.7919  
                    Weighted Avg F1-score  0.7919

3_Phosphorus


RF     SVM     KNN     MLP  \
                                                                            
Class Not Adequated Precision              0.7573  0.7399  0.6302  0.7407   
                    Recall                 0.7803  0.7750  0.8118  0.7592   
                    F1-score               0.7686  0.7571  0.7096  0.7498   
Class Adequated     Precision              0.7681  0.7473  0.7225  0.7426   
                    Recall                 0.7591  0.7279  0.7018  0.7474   
                    F1-score               0.7636  0.7375  0.7120  0.7450   
Class Excess        Precision              0.7693  0.7500  0.8272  0.7527   
                    Recall                 0.7552  0.7343  0.6139  0.7291   
                    F1-score               0.7622  0.7421  0.7047  0.7407   
Overall Performance Accuracy               0.7648  0.7456  0.7090  0.7452   
                    Macro Avg F1-score     0.7648  0.7455  0.7088  0.7452   
                    Weighted Avg F1-score  0.7648  0.7455  0.7088  0.7452   

                                              XGB  
                                                   
Class Not Adequated Precision              0.7930  
                    Recall                 0.8066  
                    F1-score               0.7997  
Class Adequated     Precision              0.7897  
                    Recall                 0.7969  
                    F1-score               0.7933  
Class Excess        Precision              0.8051  
                    Recall                 0.7840  
                    F1-score               0.7944  
Overall Performance Accuracy               0.7958  
                    Macro Avg F1-score     0.7958  
                    Weighted Avg F1-score  0.7958

3_Potassium


RF     SVM     KNN     MLP  \
                                                                            
Class Not Adequated Precision              0.8351  0.7913  0.7247  0.7795   
                    Recall                 0.8110  0.7861  0.7808  0.7703   
                    F1-score               0.8229  0.7887  0.7517  0.7749   
Class Adequated     Precision              0.7811  0.7378  0.7669  0.7409   
                    Recall                 0.8188  0.7484  0.7249  0.7419   
                    F1-score               0.7995  0.7430  0.7453  0.7414   
Class Excess        Precision              0.8115  0.7635  0.7493  0.7406   
                    Recall                 0.7955  0.7575  0.7326  0.7484   
                    F1-score               0.8034  0.7605  0.7409  0.7445   
Overall Performance Accuracy               0.8085  0.7640  0.7461  0.7535   
                    Macro Avg F1-score     0.8086  0.7641  0.7460  0.7536   
                    Weighted Avg F1-score  0.8086  0.7640  0.7460  0.7536   

                                              XGB  
                                                   
Class Not Adequated Precision              0.8530  
                    Recall                 0.8451  
                    F1-score               0.8490  
Class Adequated     Precision              0.8089  
                    Recall                 0.8279  
                    F1-score               0.8183  
Class Excess        Precision              0.8351  
                    Recall                 0.8231  
                    F1-score               0.8290  
Overall Performance Accuracy               0.8320  
                    Macro Avg F1-score     0.8321  
                    Weighted Avg F1-score  0.8321

In [ ]:


ROOT = "../Resultados/"
# Crear ventana temporal para seleccionar archivo
root = tk.Tk()
root.withdraw()  # Ocultar ventana principal

# Abrir diálogo para seleccionar archivo CSV
file_path = filedialog.askopenfilename(
    initialdir=ROOT,
    title="Seleccionar archivo CSV",
    filetypes=(("CSV files", "*.csv"), ("All files", "*.*"))
)

# Extraer el directorio del archivo seleccionado
# dir_path = os.path.dirname(file_path) if file_path else ROOT + "classification_exclude_prod_nested"
df_results = pd.read_csv(file_path)


## Procesamiento Automático de Todos los modelos

In [15]:
def process_npk_models(df_results):
    """
    Procesa resultados de modelos NPK (sin cuartiles).
    Crea una tabla organizada con modelos en filas y nutrientes en columnas.
    
    Args:
        df_results (DataFrame): DataFrame con resultados de modelos NPK
        
    Returns:
        DataFrame: Tabla organizada con accuracies por modelo y nutriente
    """
    # Extraer el nutriente de N_Classes
    df_results['Nutriente'] = df_results['N_Classes'].str.replace('3_', '')

    # Crear tabla pivotada
    tabla_organizada = df_results.pivot_table(
        index='Model',
        columns='Nutriente',
        values='Accuracy_Test',
        aggfunc='max'
    )

    # Resetear index para que Model sea una columna
    tabla_organizada = tabla_organizada.reset_index()

    # Renombrar columnas según el nutriente
    rename_dict = {
        'Model': 'Nombre Alg',
        'Nitrogen': 'Accuracy N',
        'Phosphorus': 'Accuracy P',
        'Potassium': 'Accuracy K'
    }
    tabla_organizada.rename(columns=rename_dict, inplace=True)

    # Ordenar por el promedio de accuracies
    tabla_organizada['Promedio'] = tabla_organizada[['Accuracy N', 'Accuracy P', 'Accuracy K']].mean(axis=1)
    tabla_organizada = tabla_organizada.sort_values('Promedio', ascending=False).drop('Promedio', axis=1)

    # Limpiar index y nombre de columnas
    tabla_organizada = tabla_organizada.reset_index(drop=True)
    tabla_organizada.columns.name = None

    return tabla_organizada

In [16]:
def process_quartiles_models(df_results):
    """
    Procesa resultados de modelos de cuartiles.
    Crea una tabla organizada con modelos en filas y accuracy de cuartiles.
    
    Args:
        df_results (DataFrame): DataFrame con resultados de modelos de cuartiles
        
    Returns:
        DataFrame: Tabla organizada con accuracies por modelo
    """
    # Extraer el nutriente de N_Classes
    df_results['Nutriente'] = df_results['N_Classes'].str.replace('2_', '')

    # Crear tabla pivotada
    tabla_organizada = df_results.pivot_table(
        index='Model',
        columns='Nutriente',
        values='Accuracy_Test',
        aggfunc='max'
    )

    # Resetear index para que Model sea una columna
    tabla_organizada = tabla_organizada.reset_index()

    # Renombrar columnas según el nutriente
    rename_dict = {
        'Model': 'Nombre Alg',
        'Quartiles': 'Accuracy Quartiles',
    }
    tabla_organizada.rename(columns=rename_dict, inplace=True)

    # Ordenar por el promedio de accuracies
    tabla_organizada['Promedio'] = tabla_organizada[['Accuracy Quartiles']].mean(axis=1)
    tabla_organizada = tabla_organizada.sort_values('Promedio', ascending=False).drop('Promedio', axis=1)

    # Limpiar index y nombre de columnas
    tabla_organizada = tabla_organizada.reset_index(drop=True)
    tabla_organizada.columns.name = None

    return tabla_organizada

In [17]:
# Iterar sobre todas las subcarpetas de Resultados y procesar
ROOT = "../Resultados/"

# Crear carpeta de salida
output_dir = os.path.join(ROOT, "metricas_finales")
os.makedirs(output_dir, exist_ok=True)

# Listar todas las subcarpetas
subcarpetas = [f for f in os.listdir(ROOT) if os.path.isdir(os.path.join(ROOT, f))]

resultados_procesados = []

for subcarpeta in subcarpetas:
    # Ignorar la carpeta de salida
    if subcarpeta == "metricas_finales":
        continue
    
    subcarpeta_path = os.path.join(ROOT, subcarpeta)
    
    # Buscar archivos CSV que contengan "resultados" y "completos" en el nombre
    csv_files = [f for f in os.listdir(subcarpeta_path) 
                 if f.endswith('.csv') and 'resultados' in f.lower() and 'completos' in f.lower()]
    
    if not csv_files:
        print(f"No se encontró archivo de resultados en: {subcarpeta}")
        continue
    
    # Usar el primer archivo encontrado
    csv_file = csv_files[0]
    csv_path = os.path.join(subcarpeta_path, csv_file)
    
    try:
        # Leer el CSV
        df_results = pd.read_csv(csv_path)
        
        # Determinar si es cuartiles o NPK
        is_quartiles = "cuartiles" in subcarpeta.lower()
        
        # Procesar según el tipo
        if is_quartiles:
            print(f"Procesando CUARTILES: {subcarpeta}")
            tabla_organizada = process_quartiles_models(df_results.copy())
        else:
            print(f"Procesando NPK: {subcarpeta}")
            tabla_organizada = process_npk_models(df_results.copy())
        
        # Guardar resultado
        output_filename = f"metricas_{subcarpeta}.csv"
        output_path = os.path.join(output_dir, output_filename)
        tabla_organizada.to_csv(output_path, index=False)
        
        print(f"Guardado en: {output_filename}\n")
        
        # Guardar información para resumen
        resultados_procesados.append({
            'Subcarpeta': subcarpeta,
            'Tipo': 'Cuartiles' if is_quartiles else 'NPK',
            'Archivo_Original': csv_file,
            'Archivo_Salida': output_filename,
            'Num_Modelos': len(tabla_organizada)
        })
        
    except Exception as e:
        print(f"ERROR procesando {subcarpeta}: {str(e)}\n")

# Crear resumen
df_resumen = pd.DataFrame(resultados_procesados)
resumen_path = os.path.join(output_dir, "_resumen_procesamiento.csv")
df_resumen.to_csv(resumen_path, index=False)

df_resumen

Procesando CUARTILES: classification_cuartiles_all_models
Guardado en: metricas_classification_cuartiles_all_models.csv

Procesando CUARTILES: classification_cuartiles_exclude_prod
Guardado en: metricas_classification_cuartiles_exclude_prod.csv

Procesando CUARTILES: classification_cuartiles_exclude_prod_nested
Guardado en: metricas_classification_cuartiles_exclude_prod_nested.csv

Procesando CUARTILES: classification_cuartiles_less_vars
Guardado en: metricas_classification_cuartiles_less_vars.csv

Procesando CUARTILES: classification_cuartiles_less_vars_all_models
Guardado en: metricas_classification_cuartiles_less_vars_all_models.csv

Procesando CUARTILES: classification_cuartiles_less_vars_nested
Guardado en: metricas_classification_cuartiles_less_vars_nested.csv

No se encontró archivo de resultados en: classification_cuartiles_less_vars_pca
Procesando CUARTILES: classification_cuartiles_pca
Guardado en: metricas_classification_cuartiles_pca.csv

Procesando NPK: classification_excl

,Subcarpeta,Tipo,Archivo_Original,Archivo_Salida,Num_Modelos
0,classification_cuartiles_all_models,Cuartiles,resultados_modelos_completos.csv,metricas_classification_cuartiles_all_models.csv,5
1,classification_cuartiles_exclude_prod,Cuartiles,resultados_modelos_completos.csv,metricas_classification_cuartiles_exclude_prod...,5
2,classification_cuartiles_exclude_prod_nested,Cuartiles,resultados_modelos_completos.csv,metricas_classification_cuartiles_exclude_prod...,5
3,classification_cuartiles_less_vars,Cuartiles,resultados_modelos_completos.csv,metricas_classification_cuartiles_less_vars.csv,5
4,classification_cuartiles_less_vars_all_models,Cuartiles,resultados_modelos_completos.csv,metricas_classification_cuartiles_less_vars_al...,5
5,classification_cuartiles_less_vars_nested,Cuartiles,resultados_modelos_completos.csv,metricas_classification_cuartiles_less_vars_ne...,5
6,classification_cuartiles_pca,Cuartiles,resultados_modelos_completos.csv,metricas_classification_cuartiles_pca.csv,5
7,classification_exclude_prod,NPK,resultados_modelos_completos.csv,metricas_classification_exclude_prod.csv,5
8,classification_exclude_prod_all_models_NPK,NPK,resultados_modelos_completos.csv,metricas_classification_exclude_prod_all_model...,5
9,classification_exclude_prod_nested,NPK,resultados_modelos_completos.csv,metricas_classification_exclude_prod_nested.csv,5
